# 04 양서연 · 표준링크·Vertex·도로선 구축
표준링크 매핑정보는 Selenium 화면 다운로드, 링크 Vertex는 서울 열린데이터광장 `LinkVerInfo` API로 수집한다. 로그인·캡차·차단을 우회하지 않는다.

## 1. 환경·프로젝트 루트·경로 — 네트워크·저장 없음

In [ ]:
import importlib.util, os, sys
from pathlib import Path
import pandas as pd

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate/'README.md').is_file() and (candidate/'.git').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = Path(os.getenv('SEOUL_TRAFFIC_DATA_ROOT', r'D:\\seoul-weather-traffic-data'))
TOPIS_CSV_DIR = DATA_ROOT/'topis_speed'/'csv'
ROAD_LINK_DIR = DATA_ROOT/'road_links'
VERTEX_DIR = DATA_ROOT/'vertices'
print('Python:', sys.executable)
print('프로젝트:', PROJECT_ROOT)
print('데이터:', DATA_ROOT)


## 2. 패키지·API 키 이름 확인 — 설치·키 출력 없음

In [ ]:
required_packages = ['pandas','requests','selenium']
optional_spatial = ['geopandas','shapely','pyproj','folium']
print('필수:', {name: importlib.util.find_spec(name) is not None for name in required_packages})
print('공간분석:', {name: importlib.util.find_spec(name) is not None for name in optional_spatial})
print('서울 API 키 설정 여부:', bool(os.getenv('SEOUL_OPEN_DATA_API_KEY')))


## 3. 수집 함수 불러오기 — 네트워크·브라우저 실행 없음

In [ ]:
from crawling.yang_seoyeon_link_coords import download_standard_link_mapping, fetch_link_vertices, vertices_to_lines
print('양서연 수집 함수 준비 완료')


## 4. 표준링크 매핑 다운로드 — **Selenium 브라우저·네트워크·파일 저장 발생**
전용 Chrome이 표시된다. 실행 중 해당 창을 조작하지 않는다. 기존 다운로드가 있다면 이 셀을 다시 실행하지 않는다.

In [ ]:
ROAD_LINK_DIR.mkdir(parents=True, exist_ok=True)
existing_mapping = list(ROAD_LINK_DIR.glob('*'))
if existing_mapping:
    raise FileExistsError(f'기존 파일을 보존합니다: {existing_mapping}')
mapping_path = download_standard_link_mapping(ROAD_LINK_DIR, headless=False, progress=print)
print('표준링크 저장:', mapping_path)


## 5. TOPIS 고유 링크 ID 확인 — 로컬 CSV 읽기만 발생
40개월 전체의 합집합을 사용해 특정 월에만 등장하는 링크가 빠지지 않게 한다.

In [ ]:
topis_paths = sorted(TOPIS_CSV_DIR.glob('topis_speed_*.csv'))
if len(topis_paths) != 40:
    raise RuntimeError(f'TOPIS CSV 40개가 필요합니다. 현재 {len(topis_paths)}개입니다.')
link_ids = set()
for path in topis_paths:
    ids = pd.read_csv(path, encoding='utf-8-sig', usecols=['링크아이디'], dtype='string')['링크아이디'].dropna().str.strip()
    link_ids.update(ids[ids.ne('')].tolist())
print('고유 링크 ID:', len(link_ids))


## 6. Vertex API 수집 — **API 요청 발생, 아직 저장 없음**
환경변수 `SEOUL_OPEN_DATA_API_KEY`가 필요하다. 실패 링크를 성공 행과 분리해 확인한다. API 키 값은 출력하지 않는다.

In [ ]:
vertices, vertex_failures = fetch_link_vertices(sorted(link_ids), progress=print)
print('Vertex 행:', len(vertices))
print('성공 링크:', vertices['link_id'].nunique())
print('실패 링크:', len(vertex_failures))
display(vertex_failures.head())


## 7. Vertex 원자료 저장 — **파일 저장 발생**
기존 결과는 덮어쓰지 않는다. 실패가 있어도 별도 CSV로 남긴다.

In [ ]:
VERTEX_DIR.mkdir(parents=True, exist_ok=True)
vertex_path = VERTEX_DIR/'yang_seoyeon_link_vertices_raw.csv'
failure_path = VERTEX_DIR/'yang_seoyeon_link_vertices_failures.csv'
if vertex_path.exists() or failure_path.exists():
    raise FileExistsError('기존 Vertex 결과를 덮어쓰지 않습니다.')
vertices.to_csv(vertex_path, encoding='utf-8-sig', index=False)
vertex_failures.to_csv(failure_path, encoding='utf-8-sig', index=False)
print('저장:', vertex_path, failure_path, sep='\n')


## 8. 도로 LineString 복원 — 좌표계 확인 후 실행
`SOURCE_EPSG`는 서울 API 출처 문서에서 확인한 값만 입력한다. 추정값으로 변환하지 않는다.

In [ ]:
SOURCE_EPSG = None  # 공식 문서 확인 후 정수로 입력
if SOURCE_EPSG is None:
    raise ValueError('GRS80 TM의 정확한 EPSG를 확인해 SOURCE_EPSG에 입력하세요.')
road_lines = vertices_to_lines(vertices, source_epsg=SOURCE_EPSG)
print('복원 링크 수:', len(road_lines))
display(road_lines.head())


## 9. 공간자료 저장 — **파일 저장 발생**

In [ ]:
line_path = VERTEX_DIR/'yang_seoyeon_road_lines.geojson'
if line_path.exists():
    raise FileExistsError(f'기존 파일을 덮어쓰지 않습니다: {line_path}')
road_lines.to_file(line_path, driver='GeoJSON')
print('저장:', line_path)
